# PoC — Bronze PII tokenization (Bonus Topic A)

Spike for **deterministic HMAC tokens** on fields embedded in **LLM request metadata** before rows are broadly queryable. In production the secret lives in **KMS**; analysts never see raw `user_id` / email in Silver.

**Run:** open in Jupyter or `jupyter execute submission/bonus/poc/bronze_pii_spike.ipynb`. Requires Python 3.10+ (stdlib only).


In [1]:
import hashlib
import hmac
import json
from typing import Any

# POC ONLY — use KMS-wrapped key + rotation in real Bronze writers
SALT = b"lab18-poc-rotate-in-kms-in-prod"  # noqa: S105


def tokenize_pii(value: str, purpose: str = "user") -> str:
    """Opaque token, stable for identical inputs (joinable in Silver)."""
    msg = f"{purpose}:{value}".encode()
    digest = hmac.new(SALT, msg, hashlib.sha256).hexdigest()[:16]
    return f"tok_{digest}"


def scrub_llm_record(obj: dict[str, Any]) -> dict[str, Any]:
    """Example scrub before Delta Bronze append — extend with NER / DLQ."""
    out = dict(obj)
    uid = out.get("user_id")
    if isinstance(uid, str) and uid:
        out["user_id"] = tokenize_pii(uid)
    email = out.get("email")
    if isinstance(email, str) and "@" in email:
        out["email"] = tokenize_pii(email, purpose="email")
    return out


samples = [
    {"request_id": "r1", "user_id": "alice-saas-42", "email": "alice@example.com", "model": "claude-haiku"},
    {"request_id": "r2", "user_id": "alice-saas-42", "email": "alice@example.com", "model": "claude-haiku"},
    {"request_id": "r3", "user_id": "bob-99", "email": None, "model": "claude-opus"},
]

cleaned = [scrub_llm_record(dict(row)) for row in samples]
print(json.dumps(cleaned, indent=2))

assert cleaned[0]["user_id"] == cleaned[1]["user_id"]
assert cleaned[0]["user_id"] != samples[0]["user_id"]
print("OK: stable tokens; raw PII not echoed.")


[
  {
    "request_id": "r1",
    "user_id": "tok_cc168f29895daa05",
    "email": "tok_5c807f2b8fbe6519",
    "model": "claude-haiku"
  },
  {
    "request_id": "r2",
    "user_id": "tok_cc168f29895daa05",
    "email": "tok_5c807f2b8fbe6519",
    "model": "claude-haiku"
  },
  {
    "request_id": "r3",
    "user_id": "tok_a85b0ecc5b1cab1e",
    "email": null,
    "model": "claude-opus"
  }
]
OK: stable tokens; raw PII not echoed.


## Tie-in to ARCHITECTURE.md

- Runs **without** Spark or Delta — proves the **hard part** (deterministic tokenization contract) is trivial to unit test and embed in Flink `map` before `write_deltalake`.
- Pair with **OpenLineage** facet documenting `pii:redaction=hmac_v1` on the Bronze dataset for audit.
